# 01. Interval_Report

In [41]:
# 필요 라이브러리
import asyncio, json, argparse
from datetime import datetime, timezone, timedelta
from typing import Any
from app.langgraph.common.llm import call_report_llm
from app.langgraph.common.utils import truncate_by_tokens
from app.report.utils import (
    get_last_10min_window,
    fetch_logs,
    build_log_llm_input,
    build_metric_queries,
    fetch_metric_time_map,
    aggregate_metrics,
    get_valid_metric_names,
    build_metric_llm_input,
    load_system_prompt,
    build_user_prompt_interval,
    build_chat_messages,
    save_interval_report
)

from app.core.config import (
    INTERVAL_REPORT_SYSTEM_PROMPT_PATH,
)
from app.core.logging import get_interval_logger

## 01. 기준 시간 설정

In [20]:
# 한국 표준시(KST, UTC+9) 타임존 객체 생성
# datetime 처리 시 기준 시간을 KST로 맞추기 위해 사용
kst = timezone(timedelta(hours=9))

now = datetime.now(kst)

# 현재 시각을 기준으로 가장 최근의 10분 단위 시간 구간(start/end)을 계산
start_time, end_time = get_last_10min_window(now)

print(f"- START TIME : {start_time}")
print(f"- END TIME : {end_time}")

now = end_time

- START TIME : 2026-04-24 10:40:00+09:00
- END TIME : 2026-04-24 10:50:00+09:00


## 02. 로그 수집

In [21]:
# 로그 서버(OpenSearch/Logstash)에서 데이터를 조회해 시간 범위에 맞는 로그만 필터링하고 정렬하여 반환
# Return 예시
# [
#     {
#         "log_time": "2026-04-24 10:10:05",
#         "log_level": "ERROR",
#         "pid": "12345",
#         "thread": "http-nio-8080-exec-1",
#         "logger": "com.example.service.UserService",
#         "log_text": "Database connection timeout",
#         "log_offset": 102938
#     },
#     {
#         "log_time": "2026-04-24 10:10:07",
#         "log_level": "WARN",
#         "pid": "12345",
#         "thread": "http-nio-8080-exec-2",
#         "logger": "com.example.controller.LoginController",
#         "log_text": "Login retry triggered",
#         "log_offset": 102950
#     }
# ]

try :
    logs = fetch_logs(now=now, start_time=start_time, end_time=end_time)
except Exception as e :
    print(e)
    # 로그 수집 실패해도 전체 파이프라인은 계속 진행
    logs = ""

print(f"Logs : {logs}")

404 Client Error: Not Found for url: http://10.101.5.160:9200/logstash-2026.04.24/_search
Logs : 


In [ ]:
# 로그 데이터를 LLM이 이해하기 쉬운 텍스트 포맷(파이프 구분 문자열)으로 변환
# Return 예시
# time|level|thread|logger|message
# 2026-04-24 10:10:05|ERROR|http-nio-8080-exec-1|com.example.service.UserService|Database connection timeout
# 2026-04-24 10:10:07|WARN|http-nio-8080-exec-2|com.example.controller.LoginController|Login retry triggered
# 2026-04-24 10:10:09|INFO|main|com.example.App|Application started

# 로그가 8192 토큰을 초과하면 제거
log_llm_input = truncate_by_tokens(build_log_llm_input(logs), max_tokens=8192)
print(log_llm_input)

time|level|thread|logger|message



## 03. 메트릭 수집

In [26]:
queries = build_metric_queries()
metric_names = list(queries.keys())

# 시계열 데이터 조회 (1초 단위)
time_map = fetch_metric_time_map(
    queries=queries,
    start_time=start_time,
    end_time=end_time,
    step="1s",
)

# 10초 단위로 집계
aggregated_map = aggregate_metrics(
    time_map=time_map,
    metric_names=metric_names,
    bucket_seconds=10,
)

# 데이터 품질 기준으로 metric 필터링 (null 비율 기준)
valid_metric_names = get_valid_metric_names(
    aggregated_map=aggregated_map,
    metric_names=metric_names,
    null_ratio_threshold=0.8,
)

# metric → LLM 입력 텍스트 생성
metric_llm_input = build_metric_llm_input(
    aggregated_map=aggregated_map,
    valid_metric_names=valid_metric_names,
    min_non_null_ratio=0.7,
)

print(metric_llm_input)

timestamp|http_error_rate|latency_p95|latency_p99|throughput|service_status|disk_pressure|network_pressure
10:40:00|null|null|null|null|1.000|0.695|0.000
10:40:10|null|null|null|null|1.000|0.695|0.000
10:40:20|null|null|null|null|1.000|0.695|0.000
10:40:30|null|null|null|null|1.000|0.695|0.000
10:40:40|null|null|null|null|1.000|0.695|0.000
10:40:50|null|null|null|null|1.000|0.695|0.000
10:41:00|null|null|null|null|1.000|0.695|0.000
10:41:10|null|null|null|null|1.000|0.695|0.000
10:41:20|null|null|null|null|1.000|0.695|0.000
10:41:30|null|null|null|null|1.000|0.695|0.000
10:41:40|null|null|null|null|1.000|0.695|0.000
10:41:50|null|null|null|null|1.000|0.695|0.000
10:42:00|null|null|null|null|1.000|0.695|0.000
10:42:10|null|null|null|null|1.000|0.695|0.000
10:42:20|null|null|null|null|1.000|0.695|0.000
10:42:30|null|null|null|null|1.000|0.695|0.000
10:42:40|null|null|null|null|1.000|0.695|0.000
10:42:50|null|null|null|null|1.000|0.695|0.000
10:43:00|null|null|null|null|1.000|0.695|0.000


## 04. 프롬프트 구성

In [30]:
# System Prompt
system_prompt = load_system_prompt(INTERVAL_REPORT_SYSTEM_PROMPT_PATH)
print(system_prompt)

You are a deterministic system monitoring analyst.

You will receive:
1. Time-series metrics aggregated at 10-second intervals
2. Application logs from the same 10-minute window

Your task is to analyze the system condition strictly within this 10-minute window.

METRIC DEFINITIONS

All pressure metrics are normalized between 0 and 1 (higher = more stress).

Some metrics are calculated using rate over time (change per second), while others represent current state levels.

- http_error_rate:
  Ratio of HTTP 5xx errors based on request rate over time.
  Reflects error behavior as a proportion of traffic, not a static count.

- latency_p95 / latency_p99:
  Response time percentiles (ms) derived from request duration distribution.
  Represent latency level at a point in time, not rate of change.

- throughput:
  Requests per second, calculated using request rate over time.
  Reflects traffic intensity and its changes.

- service_status:
  Service availability indicator (1 = UP, 0 = DOWN).


In [31]:
# User Prompt
user_prompt = build_user_prompt_interval(
    metric_llm_input=metric_llm_input,
    log_llm_input=log_llm_input
)

print(user_prompt)

Analyze the following information.

METRIC_INPUT_START
timestamp|http_error_rate|latency_p95|latency_p99|throughput|service_status|disk_pressure|network_pressure
10:40:00|null|null|null|null|1.000|0.695|0.000
10:40:10|null|null|null|null|1.000|0.695|0.000
10:40:20|null|null|null|null|1.000|0.695|0.000
10:40:30|null|null|null|null|1.000|0.695|0.000
10:40:40|null|null|null|null|1.000|0.695|0.000
10:40:50|null|null|null|null|1.000|0.695|0.000
10:41:00|null|null|null|null|1.000|0.695|0.000
10:41:10|null|null|null|null|1.000|0.695|0.000
10:41:20|null|null|null|null|1.000|0.695|0.000
10:41:30|null|null|null|null|1.000|0.695|0.000
10:41:40|null|null|null|null|1.000|0.695|0.000
10:41:50|null|null|null|null|1.000|0.695|0.000
10:42:00|null|null|null|null|1.000|0.695|0.000
10:42:10|null|null|null|null|1.000|0.695|0.000
10:42:20|null|null|null|null|1.000|0.695|0.000
10:42:30|null|null|null|null|1.000|0.695|0.000
10:42:40|null|null|null|null|1.000|0.695|0.000
10:42:50|null|null|null|null|1.000|0.69

In [33]:
# LLM에 전달할 최종 프롬프트
messages = build_chat_messages(system_prompt, user_prompt)
print(json.dumps(messages, ensure_ascii=False, indent=2))

[
  {
    "role": "system",
    "content": "You are a deterministic system monitoring analyst.\n\nYou will receive:\n1. Time-series metrics aggregated at 10-second intervals\n2. Application logs from the same 10-minute window\n\nYour task is to analyze the system condition strictly within this 10-minute window.\n\n========================\nMETRIC DEFINITIONS\n========================\n\nAll pressure metrics are normalized between 0 and 1 (higher = more stress).\n\nSome metrics are calculated using rate over time (change per second), while others represent current state levels.\n\n- http_error_rate:\n  Ratio of HTTP 5xx errors based on request rate over time.\n  Reflects error behavior as a proportion of traffic, not a static count.\n\n- latency_p95 / latency_p99:\n  Response time percentiles (ms) derived from request duration distribution.\n  Represent latency level at a point in time, not rate of change.\n\n- throughput:\n  Requests per second, calculated using request rate over time.

## 05. 보고서 생성

In [38]:
result, metadata = await call_report_llm(messages, type="interval")

# 시간 구간 정보 추가
result["timeWindow"] = {
    "start": start_time.strftime("%H:%M:%S"),
    "end": end_time.strftime("%H:%M:%S"),
}

print("- INTERVAL REPORT :\n\n",json.dumps(result, ensure_ascii=False, indent=2))

- INTERVAL REPORT :

 {
  "level": "low",
  "summary": "10분 구간에서 서비스는 정상 가동 상태이며, 주요 지표(트래픽, 지연, 오류율 등)는 측정되지 않았고, 디스크 압력은 중간 수준, 네트워크 압력은 없었습니다. 따라서 명확한 장애나 성능 저하 증거가 없으므로 낮은 수준으로 판단됩니다.",
  "events": [],
  "timeWindow": {
    "start": "10:40:00",
    "end": "10:50:00"
  }
}


## 06. 결과 저장

In [40]:
save_path = save_interval_report(result, start_time, end_time)
print(f"- SAVE PATH : {save_path}")

- SAVE PATH : ./data/report/interval/2026/04/24/1040_1050.json


# 02. Interval_Report_Backfill

In [42]:
import os
import subprocess
import argparse
import time
from datetime import datetime, timedelta, timezone
from app.core.logging import get_interval_backfill_logger

## 01. 기준 시간 설정

In [43]:
def get_yesterday_range(now=None):
    """
    어제 00:00 ~ 오늘 00:00 범위 반환
    (기본 백필 범위)
    """
    if now is None:
        now = datetime.now(kst)
        
    today_start = now.replace(hour=0, minute=0, second=0, microsecond=0)
    yesterday_start = today_start - timedelta(days=1)
    return yesterday_start, today_start

In [45]:
start, end = get_yesterday_range()

print(f"- START : {start}")
print(f"- END : {end}")

- START : 2026-04-23 00:00:00+09:00
- END : 2026-04-24 00:00:00+09:00


## 02. 누락된 10분 리포트 생성

In [46]:
def iter_windows(start, end, step=10):
    """
    start ~ end 구간을 step(분) 단위로 나누는 generator

    예:
    00:00 ~ 01:00 → 10분 단위
    → (00:00~00:10), (00:10~00:20) ...
    """
    cur = start
    while cur < end:
        nxt = cur + timedelta(minutes=step)
        yield cur, nxt
        cur = nxt

def get_file_path(start_time, end_time):
    """
    특정 interval 결과 파일 경로 생성

    ex)
    ./data/report/interval/2026/04/23/1200_1210.json
    """
    base_dir = (
        f"./data/report/interval/"
        f"{start_time.strftime('%Y')}/"
        f"{start_time.strftime('%m')}/"
        f"{start_time.strftime('%d')}"
    )
    filename = f"{start_time.strftime('%H%M')}_{end_time.strftime('%H%M')}.json"
    return os.path.join(base_dir, filename)

In [54]:
def backfill(start_dt, end_dt):
    """
    interval 리포트 백필 실행

    흐름:
    1. 시간 구간을 10분 단위로 나눔
    2. 이미 파일 있으면 skip
    3. 없으면 subprocess로 interval_report 실행
    4. 과부하 방지를 위해 sleep
    """
    for s, e in iter_windows(start_dt, end_dt):
        
        # 시간 제한 (08시 ~ 22시)
        if not (8 <= s.hour < 22):
            #print(f"SKIP (OUT OF RANGE): {s} ~ {e}")
            continue
        
        file_path = get_file_path(s, e)

        if os.path.exists(file_path):
            print(f"SKIP: {file_path}")
            continue

        print(f"RUN : {s} ~ {e}")

        subprocess.run([
            "python",
            "-m",
            "app.report.interval_report",
            "--start", s.strftime("%Y-%m-%d %H:%M:%S"),
            "--end", e.strftime("%Y-%m-%d %H:%M:%S"),
        ], check=True)

        # time.sleep(300)


In [55]:
backfill(start, end)

SKIP: ./data/report/interval/2026/04/23/0800_0810.json
SKIP: ./data/report/interval/2026/04/23/0810_0820.json
SKIP: ./data/report/interval/2026/04/23/0820_0830.json
SKIP: ./data/report/interval/2026/04/23/0830_0840.json
SKIP: ./data/report/interval/2026/04/23/0840_0850.json
SKIP: ./data/report/interval/2026/04/23/0850_0900.json
SKIP: ./data/report/interval/2026/04/23/0900_0910.json
SKIP: ./data/report/interval/2026/04/23/0910_0920.json
SKIP: ./data/report/interval/2026/04/23/0920_0930.json
SKIP: ./data/report/interval/2026/04/23/0930_0940.json
SKIP: ./data/report/interval/2026/04/23/0940_0950.json
SKIP: ./data/report/interval/2026/04/23/0950_1000.json
SKIP: ./data/report/interval/2026/04/23/1000_1010.json
SKIP: ./data/report/interval/2026/04/23/1010_1020.json
SKIP: ./data/report/interval/2026/04/23/1020_1030.json
SKIP: ./data/report/interval/2026/04/23/1030_1040.json
SKIP: ./data/report/interval/2026/04/23/1040_1050.json
SKIP: ./data/report/interval/2026/04/23/1050_1100.json
SKIP: ./da

# 03. Daily_Report

In [56]:
import asyncio
import json
import argparse
from datetime import datetime, timedelta, timezone
from typing import Any

from app.core.config import DAILY_REPORT_SYSTEM_PROMPT_PATH
from app.langgraph.common.utils import truncate_by_tokens
from app.report.utils import (
    load_and_filter_reports,
    build_llm_input,
    load_system_prompt,
    build_user_prompt_daily,
    build_chat_messages,
    save_daily_report,
    get_yesterday,
)
from app.langgraph.common.llm import call_report_llm
from app.core.logging import get_daily_logger

## 01. 기준 시간 설정

In [58]:
kst = timezone(timedelta(hours=9))

In [59]:
def get_target_date(date_str: str | None) -> datetime:
    """
    분석 대상 날짜 결정
    - 입력값 있으면 해당 날짜 사용
    - 없으면 '어제 날짜' 자동 계산
    """
    if date_str:
        return datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=kst)

    now = datetime.now(kst)
    return get_yesterday(now)

In [66]:
target_time = get_target_date("")
print(f"- TARGET DATE : {target_time}")

date_str = target_time.strftime("%Y-%m-%d")
print(f"- DATE STRING : {date_str}")

- TARGET DATE : 2026-04-23 00:00:00+09:00
- DATE STRING : 2026-04-23


## 02. 10분 리포트 불러오기

In [68]:
# 해당 날짜 interval 리포트 로드 및 필터링 ( Medium 이상만 들고옴 )
reports = load_and_filter_reports(date_str)

# LLM 입력용 텍스트 생성
llm_input_text_raw = build_llm_input(reports)

# 토큰 제한 (LLM max context 대응)
llm_input_text = truncate_by_tokens(llm_input_text_raw, max_tokens=8192)

print(llm_input_text)


[07:50:00 ~ 08:00:00] level=high | 서비스가 5분간 다운되었으며, 이후 정상화되었으나 디스크 압력이 지속적으로 높음.
  - (07:50:00 ~ 07:55:30) 서비스가 5분간 다운되었고, 07:55:30에 재가동.
  - (07:55:40 ~ 08:00:00) 디스크 압력이 0.7 이상으로 지속적으로 높음.
[22:00:00 ~ 22:10:00] level=high | 10분 동안 서비스가 다운되어 요청이 처리되지 않았으며, 서비스 상태가 0으로 표시되었습니다.
  - (22:00:50 ~ 22:10:00) 서비스가 22:00:50부터 22:10:00까지 다운 상태로 지속되었습니다.
[22:10:00 ~ 22:20:00] level=high | 10분 동안 서비스가 지속적으로 다운되어 있었습니다.
  - (22:10:00 ~ 22:20:00) 22:10부터 22:20까지 서비스가 다운되었습니다.
[22:20:00 ~ 22:30:00] level=high | 서비스가 10분 동안 지속적으로 다운되었습니다.
  - (22:20:00 ~ 22:30:00) 서비스가 10분간 다운되어 요청을 처리할 수 없었습니다.
[22:30:00 ~ 22:40:00] level=high | 10분 동안 서비스가 지속적으로 다운되었습니다.
  - (22:30:00 ~ 22:40:00) 22:30부터 22:40까지 서비스가 다운되었습니다.
[22:40:00 ~ 22:50:00] level=high | 10분 구간에서 서비스가 다운되어 정상 동작이 중단되었습니다.
  - (22:40:00 ~ 22:50:00) 서비스가 10분 동안 다운 상태였습니다.


## 03. 프롬프트 구성

In [73]:
system_prompt = load_system_prompt(DAILY_REPORT_SYSTEM_PROMPT_PATH)
print(system_prompt)

You are a deterministic daily service-report analyst.

The input contains ONLY abnormal intervals (medium/high).
Normal intervals are NOT included.

You MUST NOT assume the whole day was abnormal.

SEVERITY RULES

- "high":
  Only for real service disruption
  (e.g., service_status = 0, request failure)

- "medium":
  Only if:
  - abnormal intervals are frequent, sustained, or form a clear pattern

- "low":
  Default
  - few, short, or isolated anomalies
  - even if medium intervals exist

- IMPORTANT:
  - presence of medium ≠ overall medium
  - if unsure → low

ANALYSIS RULES

- Use ONLY provided interval reports
- Do NOT infer missing time ranges
- Missing = unknown (not normal)

- Merge similar or continuous events
- Avoid splitting into small fragments
- Prefer fewer, broader patterns

OUTPUT RULES

- Output ONLY valid JSON
- All text in Korean

- "summary":
  - max 5 sentences
  - describe overall stability, pattern, incidents, risk

- "actions":
  - exactly 3 items (P1, P2, P3)
 

In [74]:
user_prompt = build_user_prompt_daily(
    llm_input=llm_input_text
)
print(user_prompt)


Analyze the following daily events and generate a report.

[07:50:00 ~ 08:00:00] level=high | 서비스가 5분간 다운되었으며, 이후 정상화되었으나 디스크 압력이 지속적으로 높음.
  - (07:50:00 ~ 07:55:30) 서비스가 5분간 다운되었고, 07:55:30에 재가동.
  - (07:55:40 ~ 08:00:00) 디스크 압력이 0.7 이상으로 지속적으로 높음.
[22:00:00 ~ 22:10:00] level=high | 10분 동안 서비스가 다운되어 요청이 처리되지 않았으며, 서비스 상태가 0으로 표시되었습니다.
  - (22:00:50 ~ 22:10:00) 서비스가 22:00:50부터 22:10:00까지 다운 상태로 지속되었습니다.
[22:10:00 ~ 22:20:00] level=high | 10분 동안 서비스가 지속적으로 다운되어 있었습니다.
  - (22:10:00 ~ 22:20:00) 22:10부터 22:20까지 서비스가 다운되었습니다.
[22:20:00 ~ 22:30:00] level=high | 서비스가 10분 동안 지속적으로 다운되었습니다.
  - (22:20:00 ~ 22:30:00) 서비스가 10분간 다운되어 요청을 처리할 수 없었습니다.
[22:30:00 ~ 22:40:00] level=high | 10분 동안 서비스가 지속적으로 다운되었습니다.
  - (22:30:00 ~ 22:40:00) 22:30부터 22:40까지 서비스가 다운되었습니다.
[22:40:00 ~ 22:50:00] level=high | 10분 구간에서 서비스가 다운되어 정상 동작이 중단되었습니다.
  - (22:40:00 ~ 22:50:00) 서비스가 10분 동안 다운 상태였습니다.



In [75]:
# 최종 프롬프트
messages = build_chat_messages(system_prompt, user_prompt)

print(json.dumps(messages, ensure_ascii=False, indent=2))

[
  {
    "role": "system",
    "content": "You are a deterministic daily service-report analyst.\n\nThe input contains ONLY abnormal intervals (medium/high).\nNormal intervals are NOT included.\n\nYou MUST NOT assume the whole day was abnormal.\n\n========================\nSEVERITY RULES\n========================\n\n- \"high\":\n  Only for real service disruption\n  (e.g., service_status = 0, request failure)\n\n- \"medium\":\n  Only if:\n  - abnormal intervals are frequent, sustained, or form a clear pattern\n\n- \"low\":\n  Default\n  - few, short, or isolated anomalies\n  - even if medium intervals exist\n\n- IMPORTANT:\n  - presence of medium ≠ overall medium\n  - if unsure → low\n\n========================\nANALYSIS RULES\n========================\n\n- Use ONLY provided interval reports\n- Do NOT infer missing time ranges\n- Missing = unknown (not normal)\n\n- Merge similar or continuous events\n- Avoid splitting into small fragments\n- Prefer fewer, broader patterns\n\n=========

## 04. 보고서 생성

In [77]:
# LLM 호출 (daily 타입)
result, metadata = await call_report_llm(messages, type="daily")

# 결과에 날짜 추가
result["report_date"] = date_str

print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "overall_level": "high",
  "summary": "오늘은 07:50~08:00과 22:00~22:50 사이에 두 차례의 심각한 서비스 중단이 발생했습니다. 07:50~07:55:30 동안 5분간 다운이 있었고, 이후 디스크 압력이 지속적으로 높아졌습니다. 밤 10시부터 10분 단위로 연속된 50분간 서비스가 중단되었으며, 이는 서비스 상태가 0으로 표시되는 심각한 장애를 의미합니다. 이러한 패턴은 시스템의 안정성에 큰 위험을 초래하며, 비즈니스 연속성에 직접적인 영향을 미칩니다. 즉각적인 원인 분석과 예방 조치가 필요합니다.",
  "top_incident": {
    "time": "22:00:00 ~ 22:50:00",
    "level": "high",
    "summary": "연속 50분간 서비스가 중단되어 요청을 처리할 수 없었습니다.",
    "impact": "사용자 불편, 매출 손실, 신뢰도 하락"
  },
  "patterns": [
    {
      "type": "연속 10분 단위 서비스 중단",
      "summary": "22시 이후 10분 단위로 서비스가 반복적으로 다운되었습니다."
    },
    {
      "type": "디스크 압력 상승",
      "summary": "07:55:40~08:00:00 사이에 디스크 압력이 0.7 이상으로 지속적으로 높았습니다."
    }
  ],
  "timeline_compact": [
    {
      "time": "07:50:00",
      "level": "high",
      "summary": "서비스 5분간 다운 시작"
    },
    {
      "time": "07:55:30",
      "level": "low",
      "summary": "서비스 재가동"
    },
    {
      "time": "07:55:40",
      "level": "medium",
      "summary": "

## 05. 결과 저장

In [78]:
# 파일 저장
save_path = save_daily_report(result, date_str)
print(f"- SAVE PATH : {save_path}")

- SAVE PATH : ./data/report/daily/2026/04/23.json
